# 🎤 ChatTTS 语音合成测试

ChatTTS 是一个专为对话场景设计的中文语音合成模型，支持自然的语调、笑声、停顿等。

GitHub: https://github.com/2noise/ChatTTS

## 1. 安装依赖

首次运行需要安装 ChatTTS 及其依赖（模型约 1GB，首次会自动下载）

In [26]:
# 安装 ChatTTS和必要依赖（⚠️ transformers 必须用 4.44.x，5.x 不兼容）
# !pip install ChatTTS soundfile numpy IPython "transformers==4.44.2"

## 2. 初始化 ChatTTS 模型

首次运行会从 HuggingFace 下载模型（约 1GB），之后会使用缓存。

In [27]:
import ChatTTS
import torch
import soundfile as sf
import numpy as np
from IPython.display import Audio, display

# 初始化 ChatTTS
chat = ChatTTS.Chat()
chat.load(compile=False)  # compile=True 可以加速但首次编译较慢

print("✅ ChatTTS 模型加载完成！")

✅ ChatTTS 模型加载完成！


## 3. 基础测试 - 单句合成

In [34]:
# 基础测试：合成一句话
text = "这个牌，要不起"

wavs = chat.infer(text)

# 播放音频
audio_data = wavs[0]  # 第一个结果
if audio_data.ndim > 1:
    audio_data = audio_data.flatten()

print(f"文本: {text}")
print(f"音频长度: {len(audio_data)} samples, 采样率: 24000 Hz")
print(f"时长: {len(audio_data)/24000:.2f} 秒")
display(Audio(audio_data, rate=24000))

text:   3%|▎         | 11/384(max) [00:00, 44.62it/s]
code:   3%|▎         | 53/2048(max) [00:00, 70.88it/s]


文本: 这个牌，要不起
音频长度: 26413 samples, 采样率: 24000 Hz
时长: 1.10 秒


## 4. 使用固定音色 (Speaker Embedding)

ChatTTS 可以随机采样一个音色，也可以固定音色以保持一致性。

In [35]:
# 随机采样一个固定音色
rand_spk = chat.sample_random_speaker()
print(f"音色 embedding 长度: {len(rand_spk) if isinstance(rand_spk, (list, np.ndarray)) else type(rand_spk)}")

# 使用固定音色生成
params_infer = ChatTTS.Chat.InferCodeParams(
    spk_emb=rand_spk,  # 固定音色
    temperature=0.3,     # 较低温度 = 更稳定
    top_P=0.7,
    top_K=3,
)

# 测试同一音色说不同的话
test_texts = ["这个牌要不起", "王炸！无敌了！", "不叫，这手牌不太行。"]

for text in test_texts:
    wavs = chat.infer(text, params_infer_code=params_infer)
    audio_data = wavs[0].flatten() if wavs[0].ndim > 1 else wavs[0]
    print(f"\n🎵 {text}")
    display(Audio(audio_data, rate=24000))

音色 embedding 长度: <class 'str'>


text:   2%|▏         | 9/384(max) [00:00, 49.06it/s]
code:   3%|▎         | 55/2048(max) [00:00, 71.43it/s]



🎵 这个牌要不起


found invalid characters: {'！'}
text:   3%|▎         | 10/384(max) [00:00, 50.58it/s]
code:   2%|▏         | 50/2048(max) [00:00, 68.36it/s]



🎵 王炸！无敌了！


text:   5%|▌         | 21/384(max) [00:00, 59.38it/s]
code:   7%|▋         | 138/2048(max) [00:01, 73.46it/s]



🎵 不叫，这手牌不太行。


## 5. 控制语调和情感

ChatTTS 支持特殊标记来控制笑声、停顿等：
- `[laugh]` — 笑声
- `[uv_break]` — 停顿
- `[oral_X]` — 口语化程度 (0-9)
- `[speed_X]` — 语速 (1-9)
- `[break_X]` — 停顿时间 (ms)

In [9]:
# 带情感标记的文本
params_refine = ChatTTS.Chat.RefineTextParams(
    prompt='[oral_5][laugh_0][break_6]',  # 口语化程度2, 无笑声, 中等停顿
)

emotional_texts = [
    "炸弹来了！",           # 开心 + 笑声
    "王炸[uv_break],无敌了！",           # 霸气 + 停顿
    "不叫，这手牌不太行啊。",    # 无奈
    "三分到底！必胜之局！",              # 自信
]

for text in emotional_texts:
    wavs = chat.infer(
        text,
        params_infer_code=params_infer,
        params_refine_text=params_refine,
    )
    audio_data = wavs[0].flatten() if wavs[0].ndim > 1 else wavs[0]
    print(f"\n🎵 {text}")
    display(Audio(audio_data, rate=24000))

found invalid characters: {'！'}
text:   2%|▏         | 7/384(max) [00:00, 43.07it/s]
code:   2%|▏         | 49/2048(max) [00:00, 70.92it/s]



🎵 炸弹来了！


found invalid characters: {'！'}
text:   3%|▎         | 10/384(max) [00:00, 49.89it/s]
code:   3%|▎         | 68/2048(max) [00:00, 68.06it/s]



🎵 王炸[uv_break],无敌了！


text:   3%|▎         | 13/384(max) [00:00, 45.60it/s]
code:   4%|▍         | 89/2048(max) [00:01, 69.46it/s]



🎵 不叫，这手牌不太行啊。


found invalid characters: {'！'}
text:   3%|▎         | 13/384(max) [00:00, 49.89it/s]
code:   5%|▍         | 93/2048(max) [00:01, 67.45it/s]



🎵 三分到底！必胜之局！


## 6. 批量生成斗地主游戏语音

为斗地主游戏的所有对话生成语音文件，替换之前 macOS `say` 生成的版本。

In [30]:
import os
import json
import hashlib
from pathlib import Path

# 所有斗地主游戏对话（丰富版，约60条）
dialogues = [
    # === 叫地主阶段 ===
    "你们来吧", "这手牌一般", "你们争吧", "太烂了不叫",
    # === 叫地主阶段 - 叫一分 ===
    "试试看", "叫一分", "我来试试",
    # === 叫地主阶段 - 叫两分 ===
    "好牌必须叫", "两分", "这手牌不错", "有点意思", "两分拿下",
    # === 叫地主阶段 - 叫三分 ===
    "这把稳了",
    # === 出牌阶段 - 不要/跳过 ===
    "不要", "要不起", "打不过",
    # === 出牌 - 炸弹 ===
    "炸弹",
    # === 出牌 - 王炸/火箭 ===
    "王炸", "双王出击",
    # === 出牌 - 顺子/连牌 ===
    "顺子走起",
    # === 出牌 - 大牌压制（K，A，2） ===
     "压你一手", "压死",
    # === 出牌 - 小牌试探(小于等于6) ===
    "出张小牌", "试探一下", "小牌开路",
    # === 出牌 - 普通跟牌（大于6， 小于Q） ===
    "跟上", "出牌", "来了", "接着", "该我了",
]

print(f"共 {len(dialogues)} 条语音需要生成")

共 30 条语音需要生成


In [31]:
# ====== 音频后处理函数 ======
def postprocess_audio(audio, sr=24000, 
                      silence_threshold=0.01,   # 静音判定阈值
                      fade_out_ms=80,            # 淡出时长(毫秒)
                      tail_silence_ms=100,       # 尾部补静音(毫秒)
                      min_duration_ms=300):      # 最短时长(毫秒)
    """
    后处理音频：裁剪尾部噪声 → 淡出 → 补静音
    解决 ChatTTS 生成音频结尾不自然的问题
    """
    # 1. 裁剪尾部静音/噪声：从后向前找到最后一个有效音频位置
    frame_size = int(sr * 0.02)  # 20ms 一帧
    min_samples = int(sr * min_duration_ms / 1000)
    
    end_idx = len(audio)
    for pos in range(len(audio) - frame_size, min_samples, -frame_size):
        frame_energy = np.sqrt(np.mean(audio[pos:pos+frame_size] ** 2))
        if frame_energy > silence_threshold:
            end_idx = min(pos + frame_size * 3, len(audio))  # 多保留几帧
            break
    
    audio = audio[:end_idx]
    
    # 2. 添加淡出效果
    fade_samples = int(sr * fade_out_ms / 1000)
    fade_samples = min(fade_samples, len(audio) // 4)  # 不超过总长的1/4
    if fade_samples > 0:
        fade_curve = np.linspace(1.0, 0.0, fade_samples)
        audio[-fade_samples:] *= fade_curve
    
    # 3. 尾部补一小段静音，避免突然截断
    silence_samples = int(sr * tail_silence_ms / 1000)
    audio = np.concatenate([audio, np.zeros(silence_samples)])
    
    return audio

print("✅ 后处理函数已定义")
print("  - silence_threshold: 静音判定阈值 (越小越严格)")
print("  - fade_out_ms: 淡出时长 (毫秒)")
print("  - tail_silence_ms: 尾部补静音 (毫秒)")
print("  - min_duration_ms: 最短保留时长 (毫秒)")

✅ 后处理函数已定义
  - silence_threshold: 静音判定阈值 (越小越严格)
  - fade_out_ms: 淡出时长 (毫秒)
  - tail_silence_ms: 尾部补静音 (毫秒)
  - min_duration_ms: 最短保留时长 (毫秒)


In [32]:
# 选择一个固定音色用于所有语音
# 配置推理参数
game_infer_params = ChatTTS.Chat.InferCodeParams(
    spk_emb=rand_spk,
    temperature=0.1,
    top_P=0.7,
    top_K=5,
)

game_refine_params = ChatTTS.Chat.RefineTextParams(
    prompt='[oral_2][laugh_0][break_4]',
)

# 输出目录
output_dir = Path("assets/voices_chattts")
output_dir.mkdir(parents=True, exist_ok=True)

results = {}
failed = []

for i, text in enumerate(dialogues, 1):
    try:
        # 生成文件名（与原系统一致）
        text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()[:8]
        clean_text = ''.join(c for c in text if c.isalnum() or c in '-_')[:20]
        filename = f"{clean_text}_{text_hash}.wav"
        filepath = output_dir / filename
        
        # 生成语音
        wavs = chat.infer(
            text,
            params_infer_code=game_infer_params,
            params_refine_text=game_refine_params,
        )
        
        audio_data = wavs[0].flatten() if wavs[0].ndim > 1 else wavs[0]
        
        # ✨ 后处理：裁剪尾部 + 淡出 + 补静音
        raw_len = len(audio_data) / 24000
        audio_data = postprocess_audio(audio_data, sr=24000)
        new_len = len(audio_data) / 24000
        
        # 保存为 WAV (ChatTTS 输出 24000Hz)
        sf.write(str(filepath), audio_data, 24000)
        
        results[text] = f"./assets/voices_chattts/{filename}"
        print(f"  [{i:2d}/{len(dialogues)}] ✅ {text} → {filename} ({raw_len:.1f}s → {new_len:.1f}s)")
        
    except Exception as e:
        failed.append(text)
        print(f"  [{i:2d}/{len(dialogues)}] ❌ {text} — 错误: {e}")

print(f"\n🎉 完成！成功: {len(results)}/{len(dialogues)}")
if failed:
    print(f"失败: {failed}")

text:   1%|▏         | 5/384(max) [00:00, 28.41it/s]
code:   2%|▏         | 40/2048(max) [00:00, 67.87it/s]


  [ 1/30] ✅ 你们来吧 → 你们来吧_f952affc.wav (0.8s → 0.8s)


text:   2%|▏         | 6/384(max) [00:00, 42.18it/s]
code:   2%|▏         | 50/2048(max) [00:00, 62.07it/s]


  [ 2/30] ✅ 这手牌一般 → 这手牌一般_14aad410.wav (1.0s → 1.1s)


text:   1%|▏         | 5/384(max) [00:00, 41.56it/s]
code:   2%|▏         | 39/2048(max) [00:00, 64.59it/s]


...

  [28/30] ✅ 来了 → 来了_0424ac2f.wav (0.8s → 0.9s)


text:   1%|          | 3/384(max) [00:00, 26.23it/s]
code:   1%|          | 15/2048(max) [00:00, 40.67it/s]


  [29/30] ✅ 接着 → 接着_7c90c225.wav (0.3s → 0.4s)


text:   1%|          | 4/384(max) [00:00, 29.63it/s]
code:   1%|          | 22/2048(max) [00:00, 55.56it/s]


  [30/30] ✅ 该我了 → 该我了_f4339847.wav (0.4s → 0.5s)

🎉 完成！成功: 30/30


In [33]:
# 保存语音映射文件
mapping_file = output_dir / "voice_mapping.json"
with open(mapping_file, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✅ 映射文件已保存: {mapping_file}")
print(f"\n包含 {len(results)} 条语音映射")

✅ 映射文件已保存: assets/voices_chattts/voice_mapping.json

包含 30 条语音映射


## 7. 试听对比

对比 ChatTTS 生成的语音与之前 macOS `say` 生成的语音

In [ ]:
# 试听几条 ChatTTS 生成的语音
sample_texts = ["炸弹来了", "王炸", "不叫", "让你们来吧", "这把稳了"]

for text in sample_texts:
    text_hash = hashlib.md5(text.encode('utf-8')).hexdigest()[:8]
    clean_text = ''.join(c for c in text if c.isalnum() or c in '-_')[:20]
    filename = f"{clean_text}_{text_hash}.wav"
    
    chattts_path = output_dir / filename
    original_path = Path("assets/voices") / filename
    
    print(f"\n--- 🎵 {text} ---")
    
    if chattts_path.exists():
        data, sr = sf.read(str(chattts_path))
        print(f"ChatTTS 版本 ({len(data)/sr:.1f}s):")
        display(Audio(data, rate=sr))
    
    if original_path.exists():
        data, sr = sf.read(str(original_path))
        print(f"macOS say 版本 ({len(data)/sr:.1f}s):")
        display(Audio(data, rate=sr))

## 8. 替换游戏语音（可选）

如果对 ChatTTS 效果满意，运行下面的代码将新语音替换到游戏目录。

In [ ]:
# ⚠️ 取消注释下面的代码来替换游戏语音
# import shutil
# 
# # 备份原始语音
# backup_dir = Path("assets/voices_backup_say")
# if not backup_dir.exists():
#     shutil.copytree("assets/voices", backup_dir)
#     print(f"✅ 原始语音已备份到: {backup_dir}")
# 
# # 复制 ChatTTS 语音到游戏目录
# for wav_file in output_dir.glob("*.wav"):
#     dest = Path("assets/voices") / wav_file.name
#     shutil.copy2(wav_file, dest)
# 
# # 更新映射文件（路径改为 assets/voices）
# updated_mapping = {}
# for text, path in results.items():
#     updated_mapping[text] = path.replace("voices_chattts", "voices")
# 
# with open("assets/voices/voice_mapping.json", 'w', encoding='utf-8') as f:
#     json.dump(updated_mapping, f, ensure_ascii=False, indent=2)
# 
# print(f"✅ 已将 {len(results)} 个 ChatTTS 语音文件复制到游戏目录")
# print("🎮 刷新游戏页面即可听到新语音！")

print("取消上面的注释并运行来替换游戏语音")

## 9. 尝试不同音色

如果不满意当前音色，可以多采样几个试试。

In [24]:
# 采样5个不同音色，用同一句话试听
test_sentence = "炸弹来了！王炸！无敌了！"

print(f"测试文本: {test_sentence}\n")

for i in range(5):
    spk = chat.sample_random_speaker()
    params = ChatTTS.Chat.InferCodeParams(
        spk_emb=spk,
        temperature=0.3,
        top_P=0.7,
        top_K=20,
    )
    wavs = chat.infer(test_sentence, params_infer_code=params)
    audio_data = wavs[0].flatten() if wavs[0].ndim > 1 else wavs[0]
    print(f"音色 #{i+1}:")
    display(Audio(audio_data, rate=24000))

found invalid characters: {'！'}


测试文本: 炸弹来了！王炸！无敌了！



text:   4%|▍         | 16/384(max) [00:00, 35.37it/s]
code:   4%|▍         | 88/2048(max) [00:01, 59.52it/s]


音色 #1:


found invalid characters: {'！'}
text:   4%|▍         | 17/384(max) [00:00, 45.41it/s]
code:   4%|▍         | 92/2048(max) [00:01, 60.21it/s]


...

音色 #3:


found invalid characters: {'！'}
text:   4%|▍         | 15/384(max) [00:00, 33.19it/s]
code:   4%|▍         | 85/2048(max) [00:01, 59.02it/s]


音色 #4:


found invalid characters: {'！'}
text:   4%|▍         | 16/384(max) [00:00, 45.98it/s]
code:   4%|▍         | 86/2048(max) [00:01, 59.42it/s]


音色 #5:
